In [ ]:
from pymilvus import MilvusClient

client = MilvusClient("./milvus_demo.db", db_name="demo")
# client = MilvusClient(uri="http://localhost:19530", token="username:password")

if client.has_collection(collection_name="demo_collection"):
    client.drop_collection(collection_name="demo_collection")

client.create_collection(
    collection_name="demo_collection",
    dimension=768,  # The vectors we will use in this demo has 768 dimensions
)

# 删除 collections
# client.drop_collection(collection_name="demo_collection")
# client.drop_collection()

In [ ]:
from langchain_openai import OpenAIEmbeddings
# 超时 —— 无法执行成功
import os


# 配置硅基流动的 API
os.environ['OPENAI_API_KEY'] = os.getenv('SILICON_API_KEY')
os.environ['OPENAI_BASE_URL'] = os.getenv('SILICON_BASE_URL_OPEN_AI')

# 初始化 Embedding 模型
embedding_model = OpenAIEmbeddings(
    model=os.getenv('SILICON_EMBEDDING_MODEL', 'Qwen/Qwen3-VL-Embedding-8B'),
    openai_api_base=os.getenv('SILICON_BASE_URL_OPEN_AI')
)

docs = [
    "Artificial intelligence was founded as an academic discipline in 1956.",
    "Alan Turing was the first person to conduct substantial research in AI.",
    "Born in Maida Vale, London, Turing was raised in southern England.",
]

vectors = embedding_model.embed_documents(docs)
print("Dim:", embedding_model.dim, vectors[0].shape)  # Dim: 768 (768,)

data = [
    {"id": i, "vector": vectors[i], "text": docs[i], "subject": "history"}
    for i in range(len(vectors))
]

print("Data has", len(data), "entities, each with fields: ", data[0].keys())
print("Vector dim:", len(data[0]["vector"]))


In [ ]:
# Mock data
import random

docs = [
    "Artificial intelligence was founded as an academic discipline in 1956.",
    "Alan Turing was the first person to conduct substantial research in AI.",
    "Born in Maida Vale, London, Turing was raised in southern England.",
]
vectors = [[random.uniform(-1, 1) for _ in range(768)] for _ in docs]
data = [
    {"id": i, "vector": vectors[i], "text": docs[i], "subject": "history"}
    for i in range(len(vectors))
]

print("Data has", len(data), "entities, each with fields: ", data[0].keys())
print("Vector dim:", len(data[0]["vector"]))

In [ ]:
# 插入数据
res = client.insert(collection_name="demo_collection", data=data)

print(res)

In [ ]:
# 向量搜索
from pymilvus import model

embedding_fn = model.DefaultEmbeddingFunction()

query_vectors = embedding_fn.encode_queries(["Who is Alan Turing?"])

res = client.search(
    collection_name="demo_collection",  # target collection
    data=query_vectors,  # query vectors
    limit=2,  # number of returned entities
    output_fields=["text", "subject"],  # specifies fields to be returned
)

print(res)